In [0]:
df= spark.read.parquet("/Volumes/anac/databricks/anac_volume/02.Silver/")

In [0]:
# 1-Secionar colunas
colunas = ['Aerodromo_de_Destino', 'Aerodromo_de_Origem', 'Classificacao_da_Ocorrência', 'Danos_a_Aeronave', 'Data_da_Ocorrencia','Municipio', 'UF', 'Regiao','Tipo_de_Aerodromo','Tipo_de_Ocorrencia', 'Lesoes_Desconhecidas_Passageiros','Lesoes_Desconhecidas_Terceiros', 'Lesoes_Desconhecidas_Tripulantes', 'Lesoes_Fatais_Passageiros', 'Lesoes_Fatais_Terceiros', 'Lesoes_Fatais_Tripulantes', 'Lesoes_Graves_Passageiros', 'Lesoes_Graves_Terceiros', 'Lesoes_Graves_Tripulantes', 'Lesoes_Leves_Passageiros', 'Lesoes_Leves_Terceiros', 'Lesoes_Leves_Tripulantes']

df= df.select(colunas)


In [0]:
# 2-somar
colunas_somar = ['Lesoes_Desconhecidas_Passageiros','Lesoes_Desconhecidas_Terceiros', 'Lesoes_Desconhecidas_Tripulantes', 'Lesoes_Fatais_Passageiros', 'Lesoes_Fatais_Terceiros', 'Lesoes_Fatais_Tripulantes', 'Lesoes_Graves_Passageiros', 'Lesoes_Graves_Terceiros', 'Lesoes_Graves_Tripulantes', 'Lesoes_Leves_Passageiros', 'Lesoes_Leves_Terceiros', 'Lesoes_Leves_Tripulantes']

df = df.withColumn("Total_lesoes",sum(df[coluna] for coluna in colunas_somar))

In [0]:
#3-renomear
df= df\
    .withColumnRenamed('Aerodromo_de_Destino', 'Destino')\
    .withColumnRenamed('Aerodromo_de_Origem', 'Origem')\
    .withColumnRenamed('Classificacao_da_Ocorrência', 'Classificacao')\
    .withColumnRenamed('Danos_a_Aeronave', 'Danos')\
    .withColumnRenamed('Data_da_Ocorrencia', 'Data')\
    .withColumnRenamed('UF','Estado')

In [0]:
# 4-excluir dados
cassificacoes_excluir = ['Indeterminado','Sem Registro','Exterior']
df = df.filter(~df.Estado.isin(cassificacoes_excluir))

In [0]:
# 5-Inserir coluna data
from pyspark.sql.functions import current_timestamp,date_format,from_utc_timestamp
df = df.withColumn("Atualizacao",\
    date_format(from_utc_timestamp(current_timestamp(), "America/Sao_Paulo"),"yyyy-MM-dd HH:mm:ss"))

In [0]:
# 6-Salvar tudo na camada Gold particionada por UF > Estado
df.write\
    .mode("overwrite")\
    .format("parquet")\
    .partitionBy("Estado")\
    .save("/Volumes/anac/databricks/anac_volume/03.Gold/")